# Clase 045 — APIs REST con requests

**Parte 0** · requests docs.

> 🎯 GET/POST, status codes, auth, paginación, rate limiting, Session.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
import requests
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
print(f'requests: {requests.__version__}')

## 1️⃣ GET básico

```python
r = requests.get('https://api.github.com')
r.status_code   # 200
r.headers       # dict con metadata
r.json()        # parsea response body como JSON
r.text          # raw string
```

In [ ]:
try:
    r = requests.get('https://api.github.com', timeout=10)
    print(f'status: {r.status_code}')
    print(f'content-type: {r.headers.get("content-type")}')
    body = r.json()
    print(f'keys: {list(body.keys())[:5]}...')
except requests.RequestException as e:
    print(f'sin red: {e}')

## 2️⃣ Params y headers

```python
r = requests.get(
    'https://api.github.com/search/repositories',
    params={'q': 'python ml', 'sort': 'stars', 'per_page': 5},
    headers={'Accept': 'application/vnd.github+json'},
    timeout=10,
)
```

`params` se convierten a `?q=...&sort=...` automáticamente, con encoding seguro.

In [ ]:
try:
    r = requests.get(
        'https://api.github.com/search/repositories',
        params={'q': 'python machine learning', 'sort': 'stars', 'per_page': 5},
        timeout=10,
    )
    r.raise_for_status()
    for item in r.json()['items'][:5]:
        print(f"{item['stargazers_count']:>7,}  {item['full_name']}")
except requests.RequestException as e:
    print(f'error: {e}')

## 3️⃣ Status codes — qué significan

| Familia | Significado | Reacción típica |
|---|---|---|
| 2xx | Éxito | continuar |
| 3xx | Redirección | requests sigue automáticamente |
| 4xx | Error cliente (404, 401, 403, 429) | revisar tu request |
| 5xx | Error servidor | reintentar con backoff |

**`raise_for_status()`** lanza `HTTPError` si status >= 400:

In [ ]:
try:
    r = requests.get('https://api.github.com/this/does/not/exist', timeout=10)
    r.raise_for_status()
except requests.HTTPError as e:
    print(f'HTTP error: {e}')
    print(f'status fue: {r.status_code}')

## 4️⃣ Autenticación

```python
# Bearer token
r = requests.get(URL, headers={'Authorization': f'Bearer {TOKEN}'})

# API key en header
r = requests.get(URL, headers={'X-API-Key': KEY})

# Basic auth (legacy)
from requests.auth import HTTPBasicAuth
r = requests.get(URL, auth=HTTPBasicAuth('user', 'pw'))
```

**Regla**: tokens y keys NUNCA hardcoded en el código. Usa variables de entorno o secret manager:

```python
import os
TOKEN = os.environ['GITHUB_TOKEN']
```

## 5️⃣ Paginación

Patrones comunes:

1. **Offset/limit**: `?page=2&per_page=100`
2. **Cursor**: `?after=<id_anterior>`
3. **Link header**: la respuesta incluye `Link: <...>; rel="next"`

GitHub usa los 3 según endpoint.

In [ ]:
# Demo paginación con events (3 páginas)
try:
    total = []
    for page in range(1, 4):
        r = requests.get(
            'https://api.github.com/events',
            params={'page': page, 'per_page': 5},
            timeout=10,
        )
        if r.status_code != 200:
            print(f'page {page}: status {r.status_code}, parar')
            break
        data = r.json()
        total.extend(data)
        print(f'page {page}: {len(data)} events')
        time.sleep(0.5)   # cortesía
    print(f'total acumulado: {len(total)}')
except requests.RequestException as e:
    print(f'error: {e}')

## 6️⃣ Rate limiting + retry exponencial

```python
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

retry = Retry(
    total=3,
    backoff_factor=1.0,   # espera 1s, 2s, 4s entre intentos
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=['GET'],
)
adapter = HTTPAdapter(max_retries=retry)

session = requests.Session()
session.mount('https://', adapter)
session.mount('http://', adapter)

# Ahora cualquier session.get() reintenta automáticamente en 5xx/429
r = session.get('https://api.github.com')
```

## 7️⃣ `Session` — reutilizar conexión

Mejora rendimiento al reusar el TCP/TLS handshake, y mantiene cookies entre requests:

In [ ]:
# Comparar Session vs requests directo
N = 10
url = 'https://api.github.com/zen'   # endpoint trivial

t0 = time.perf_counter()
for _ in range(N):
    try:
        requests.get(url, timeout=10)
    except Exception:
        pass
t1 = time.perf_counter()

t2 = time.perf_counter()
with requests.Session() as s:
    for _ in range(N):
        try:
            s.get(url, timeout=10)
        except Exception:
            pass
t3 = time.perf_counter()

print(f'sin session: {(t1-t0)*1000:.0f} ms')
print(f'con session: {(t3-t2)*1000:.0f} ms')
print(f'speedup    : {(t1-t0)/(t3-t2):.2f}× (sólo notable con muchas requests)')

## ✅ Checklist

- [ ] Sé GET/POST con params/headers/json
- [ ] Verifico status code (raise_for_status)
- [ ] Manejo auth con header Bearer
- [ ] Pagino con loop hasta que no haya más
- [ ] Configuro Retry para 5xx/429
- [ ] Uso Session para múltiples requests

## 📝 Homework

Ver `README.md`. API pública, manejo errores, pagination, Session+Retry, benchmark.

## 📖 Definiciones y características

**REST (REpresentational State Transfer)**

Estilo arquitectónico para APIs web sobre HTTP. Recursos identificados por URLs, operaciones por verbos HTTP (GET=leer, POST=crear, PUT=update, DELETE=borrar).

**`requests`**

Librería Python de facto para hacer HTTP. API simple: `requests.get(url, params=..., headers=..., timeout=...)`. Soporta auth, cookies, sessions, retry.

**Status code**

Número HTTP que indica resultado: **2xx** éxito, **3xx** redirect, **4xx** error cliente (404 no encontrado, 401 no auth, 403 prohibido, 429 rate limited), **5xx** error servidor.

**Paginación**

API que devuelve resultados en bloques (no todo de golpe). Patrones: **offset/limit** (`?page=2`), **cursor** (`?after=<id>`), **Link header** (`<url>; rel="next"`).

**Rate limiting**

Política del servidor: máximo N requests/seg/usuario. Excederlo → 429. **Respétalo** con delays y retries exponenciales.

**`Session`**

Reuso de conexión TCP/TLS entre requests. Mantiene cookies. ~10× más rápido para múltiples requests al mismo host vs `requests.get` repetido.

**Bearer token**

Esquema de auth común: `Authorization: Bearer <token>` header. Token suele ser JWT o opaque string emitido por OAuth/login.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `requests.exceptions.ConnectTimeout` o `Timeout` | API lenta o sin red. **Fix**: siempre pasa `timeout=10` (segundos) — sin él, request puede colgarse indefinidamente. |
| API responde 200 pero `.json()` lanza error | Body no es JSON (HTML de error, vacío). **Fix**: verifica `r.headers['content-type']` o usa `try: r.json() except ValueError: print(r.text)`. |
| Hardcodeé el token en el código y subí a GitHub | **Catástrofe de seguridad** — el token es público. **Fix**: rota el token YA, usa `.env` + `python-dotenv`, añade `.env` a `.gitignore`. |
| Mi script tira la API ajena (HTTP 429) | Sin rate limiting. **Fix**: `time.sleep()` entre requests, o `Session` con `Retry(backoff_factor=2)` para reintentos exponenciales. |
| HTTPError no se lanza con status 4xx | `requests` NO lanza por default. **Fix**: `r.raise_for_status()` después de `requests.get(...)` para lanzar en 4xx/5xx. |

## ❓ Preguntas frecuentes

**❓ ¿`requests` o `httpx`?**

**`requests`** sigue siendo la default (estable, ubícua). **`httpx`** drop-in con async support (`async with httpx.AsyncClient() as client:`). Para async/HTTP2, httpx; para todo lo demás, requests.

**❓ ¿Cuándo Session?**

Más de 2-3 requests al mismo host. La primera request hace handshake TCP/TLS (~100ms); Session lo reusa. Para single request, no aporta.

**❓ ¿`json=` o `data=` en POST?**

**`json=dict`**: serializa a JSON y setea `Content-Type: application/json`. **`data=dict`**: form-encoded (`application/x-www-form-urlencoded`). Para APIs REST modernas, casi siempre `json=`.

**❓ ¿Cómo paginar genéricamente?**

Loop hasta que la API diga "no más": `while True: r = requests.get(url, params=...); items.extend(r.json()['data']); if not r.json().get('next'): break`.

**❓ ¿Auth OAuth desde Python?**

Para casos simples (Bearer fijo): pasa el header. Para OAuth flow completo: `authlib`, `requests-oauthlib`. Para producción: librería oficial del proveedor (`google-auth`, `pyOpenSSL`, etc.).

## 🔗 Referencias

- [requests docs](https://requests.readthedocs.io/)
- [urllib3 Retry](https://urllib3.readthedocs.io/en/stable/reference/urllib3.util.html#urllib3.util.Retry)

➡️ **Siguiente:** [046 — Web scraping con BeautifulSoup](../046-web-scraping-con-beautifulsoup/README.md)